# Pilot: does distribution shift break probabilistic timing guarantees for early-exit inference?
**Ashhar Hussain**, pilot for the proposal *Characterising and Evaluating the Timing Reliability of Embedded Neural-Network Inference under Probabilistic Real-Time Scheduling*.

**Question.** An engineer profiles an early-exit CNN on in-distribution data, feeds the measured execution-time distribution into a convolution-based probabilistic response-time analysis, and picks the shortest period whose predicted deadline-miss probability is ≤ 1%. What is the **real** miss rate under noise shift, OOD inputs and bursty "smoke-episode" shift?

**How to run:** Runtime → Run all. Takes about 2–4 minutes on any runtime, **TPU included**. Inference is timed on **one CPU thread on purpose**, because the proposal is about embedded CPUs (Raspberry Pi / Jetson-class), and TPU latency would not represent them. The last cell downloads `github_upload.zip` and prints a block to paste back.

In [ ]:
# 1) Setup — works on any Colab runtime (CPU / GPU / TPU). Timing is done on ONE CPU thread on purpose (embedded-CPU study).
import os, sys, time, json, math, gc, platform, subprocess, shutil
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    import torch
import torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import fftconvolve
from scipy.stats import ks_2samp
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED, N, TAU, K, TARGET = 0, 2500, 0.90, 4, 1e-2   # N timed inferences/condition | TAU exit confidence | control task runs K x per frame | design target miss prob
REPO = "github_upload"; OUT = f"{REPO}/results"; os.makedirs(OUT, exist_ok=True)
torch.manual_seed(SEED); rng = np.random.default_rng(SEED); g = torch.Generator().manual_seed(SEED)
cpu = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
ENV = {"cpu": cpu, "cores": os.cpu_count(), "python": platform.python_version(), "torch": torch.__version__}
print(ENV)

In [ ]:
# 2) Data (bundled with scikit-learn, no download), shifted/OOD pools, early-exit CNN, training (~10-40 s)
Xn, yn = load_digits(return_X_y=True)
tr, te = train_test_split(np.arange(len(yn)), test_size=0.4, random_state=SEED, stratify=yn)
X = F.interpolate(torch.tensor(Xn, dtype=torch.float32).view(-1, 1, 8, 8) / 16, size=(32, 32), mode="bilinear", align_corners=False)
y = torch.tensor(yn, dtype=torch.long)
Xtr, ytr, Xte, yte = X[torch.as_tensor(tr)], y[torch.as_tensor(tr)], X[torch.as_tensor(te)], y[torch.as_tensor(te)]
noisy = lambda Z, s: (Z + s * torch.randn(Z.shape, generator=g)).clamp(0, 1)
POOLS = {"ID": (Xte, yte), "Shift_mild": (noisy(Xte, 0.25), yte), "Shift_strong": (noisy(Xte, 0.5), yte),
         "OOD_noise": (torch.rand(Xte.shape, generator=g), None)}

class EarlyExitCNN(nn.Module):
    def __init__(self, c=(24, 48, 96), n=10):
        super().__init__()
        blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(), nn.Conv2d(o, o, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.blocks = nn.ModuleList([blk(1, c[0]), blk(c[0], c[1]), blk(c[1], c[2])])
        self.heads = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(k, n)) for k in c])
    def forward(self, x):                                   # training: logits of all 3 exits
        out = []
        for b, h in zip(self.blocks, self.heads):
            x = b(x); out.append(h(x))
        return out
    @torch.no_grad()
    def infer(self, x, tau):                                # deployment: stop at the first confident exit
        for s, (b, h) in enumerate(zip(self.blocks, self.heads), 1):
            x = b(x); conf, pred = torch.softmax(h(x), 1).max(1)
            if conf.item() >= tau or s == len(self.blocks):
                return s, int(pred.item())
    @torch.no_grad()
    def infer_static(self, x):                              # same network, always full depth (static control)
        for b in self.blocks: x = b(x)
        return 3, int(self.heads[-1](x).argmax(1).item())

m = EarlyExitCNN(); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
torch.set_num_threads(min(8, os.cpu_count())); t0 = time.time()
for ep in range(8):
    perm = torch.randperm(len(Xtr), generator=g)
    for i in range(0, len(Xtr), 64):
        b = perm[i:i + 64]; loss = sum(F.cross_entropy(o, ytr[b]) for o in m(Xtr[b]))
        opt.zero_grad(); loss.backward(); opt.step()
m.eval(); print(f"trained in {time.time() - t0:.0f}s")

@torch.no_grad()
def batch_eval(Z, yz):
    P = [torch.softmax(o, 1) for o in m(Z)]
    conf = torch.stack([p.max(1).values for p in P], 1).numpy(); pred = torch.stack([p.argmax(1) for p in P], 1).numpy()
    ex = np.where(conf[:, 0] >= TAU, 0, np.where(conf[:, 1] >= TAU, 1, 2))
    r = {f"exit{s + 1}": float((ex == s).mean()) for s in range(3)}
    if yz is not None:
        r["acc_full"] = float((pred[:, 2] == yz.numpy()).mean())
        r["acc_early_exit"] = float((pred[np.arange(len(ex)), ex] == yz.numpy()).mean())
    return r
ACC = {k: batch_eval(*v) for k, v in POOLS.items()}
torch.set_num_threads(1)                                    # ALL timing below: one CPU thread
print(pd.DataFrame(ACC).T.round(3))

In [ ]:
# 3) Input streams + interleaved timing (~30-60 s). Conditions run in random order per round, so platform drift cannot masquerade as shift.
def stream(name, n):
    Z, yz = POOLS[name]; i = torch.as_tensor(rng.integers(0, len(Z), n))
    return Z[i], (yz[i].numpy() if yz is not None else None)
FR, LAB = {}, {}
for c, src in [("ID_profile", "ID"), ("ID", "ID"), ("Shift_mild", "Shift_mild"), ("Shift_strong", "Shift_strong"),
               ("OOD_noise", "OOD_noise"), ("Static_ID", "ID"), ("Static_strong", "Shift_strong")]:
    FR[c], LAB[c] = stream(src, N)
# Bursty shift ("smoke episodes"): 2-state Markov chain, mean episode 75 frames, ~27% of frames shifted
state = np.zeros(N, np.int8); s = 0
for i in range(N):
    s = (1 if rng.random() < 1 / 200 else 0) if s == 0 else (0 if rng.random() < 1 / 75 else 1); state[i] = s
a, la = stream("ID", N); b_, lb = stream("Shift_strong", N)
FR["Bursty"] = torch.where(torch.as_tensor(state).bool().view(-1, 1, 1, 1), b_, a); LAB["Bursty"] = np.where(state == 1, lb, la)
CONDS = list(FR)
T = {c: np.empty(N) for c in CONDS}; ST = {c: np.empty(N, np.int8) for c in CONDS}; PR = {c: np.empty(N, np.int64) for c in CONDS}
for i in range(50): m.infer(FR["ID"][i:i + 1], TAU); m.infer_static(FR["ID"][i:i + 1])     # warm-up
CH = 10; cs = N // CH; gc.collect(); gc.disable(); t0 = time.time()
for k in range(CH):
    for c in rng.permutation(CONDS):
        Z, static = FR[c], str(c).startswith("Static")
        for i in range(k * cs, (k + 1) * cs):
            x = Z[i:i + 1]; t_ = time.perf_counter_ns()
            s, p = m.infer_static(x) if static else m.infer(x, TAU)
            T[c][i] = (time.perf_counter_ns() - t_) * 1e-6; ST[c][i] = s; PR[c][i] = p
gc.enable(); print(f"timed {N * len(CONDS)} inferences in {time.time() - t0:.0f}s")

def tstats(t):
    return {"mean": t.mean(), "p50": np.percentile(t, 50), "p95": np.percentile(t, 95), "p99": np.percentile(t, 99),
            "p99.9": np.percentile(t, 99.9), "max": t.max(), "lag1_autocorr": np.corrcoef(t[:-1], t[1:])[0, 1]}
TS = pd.DataFrame({c: {**tstats(T[c]), **{f"exit{s}": (ST[c] == s).mean() for s in (1, 2, 3)},
                       "acc": (PR[c] == LAB[c]).mean() if LAB[c] is not None else np.nan,
                       "KS_D_vs_profile": ks_2samp(T[c], T["ID_profile"]).statistic} for c in CONDS}).T
STATIC = {"median_change_pct": 100 * (np.median(T["Static_strong"]) / np.median(T["Static_ID"]) - 1), "KS_D": ks_2samp(T["Static_ID"], T["Static_strong"]).statistic}
EE = {"median_change_pct": 100 * (np.median(T["Shift_strong"]) / np.median(T["ID"]) - 1), "KS_D": ks_2samp(T["ID"], T["Shift_strong"]).statistic}
print(TS.round(3)); print("strong shift -> static net:", {k: round(v, 3) for k, v in STATIC.items()}, "| early-exit net:", {k: round(v, 3) for k, v in EE.items()})

In [ ]:
# 4) Convolution-based stochastic response-time analysis + trace-driven simulator
# Task set: one core, fixed-priority preemptive, synchronous & harmonic.
#   tau1 (high priority): control task, period T2/K, execution ~ U[0.8, 1.2] * c1
#   tau2 (low priority) : early-exit inference, period = deadline = T2, execution = measured
# Analysis = stationary backlog over hyperperiods + split-convolve-merge for preemptions (the hyperperiod/backlog
# approach the TII 2026 CAN paper builds on). Simplified uniprocessor version, NOT a reproduction of the CAN algorithm.
NB = 50                                                     # grid bins per tau1 period
def conv(a, b, L):
    c = np.clip(fftconvolve(a, b), 0, None)
    if len(c) >= L:
        c[L - 1] += c[L:].sum(); return c[:L]
    return np.pad(c, (0, L - len(c)))
def pmf(x, dt, L):                                          # ceil rounding = safe (pessimistic) discretisation
    k = np.clip(np.ceil(np.asarray(x) / dt).astype(int), 0, L - 1)
    p = np.bincount(k, minlength=L).astype(float); return np.trim_zeros(p / p.sum(), "b")
def preempt(A, r, C1, L):                                   # tau1 job released at bin r delays every tau2 job still running at r
    if r >= L - 1: return A
    return np.concatenate([A[:r + 1], np.zeros(L - r - 1)]) + conv(np.concatenate([np.zeros(r + 1), A[r + 1:]]), C1, L)
C1_UNITS = np.random.default_rng(1).uniform(0.8, 1.2, 20000)
def analyse(c2_samples, c1, T2, max_iter=600, tol=1e-13):
    """Predicted stationary deadline-miss probability of tau2, assuming i.i.d. execution times."""
    if 1.2 * c1 >= T2 / K: return 1.0
    n1, n2 = NB, K * NB; dt = T2 / n2; L = 12 * n2
    C2, C1 = pmf(c2_samples, dt, L), pmf(C1_UNITS * c1, dt, L)
    W = np.zeros(L); W[0] = 1.0                             # tau2 backlog at hyperperiod start
    for it in range(max_iter):
        A = conv(conv(W, C1, L), C2, L)
        for j in range(1, K): A = preempt(A, j * n1, C1, L)
        Wn = np.zeros(L); Wn[0] = A[:n2 + 1].sum(); Wn[1:L - n2] = A[n2 + 1:]
        done = np.abs(Wn - W).sum() < tol; W = Wn
        if done: break
    R = conv(conv(W, C1, L), C2, L)
    for j in range(1, L // n1 + 1): R = preempt(R, j * n1, C1, L)
    return float(R[n2 + 1:].sum())
def simulate(c2_seq, c1, T2, seed=2):
    """Trace-driven simulation of the same task set (keeps the real job order); returns tau2 response times."""
    T1 = T2 / K; C1s = np.random.default_rng(seed).uniform(0.8, 1.2, 65536) * c1
    fin, R = 0.0, np.empty(len(c2_seq))
    for i, w in enumerate(c2_seq):
        rel = i * T2; t = max(rel, fin); j = int(t // T1)
        while True:
            if t >= (j + 1) * T1: j += 1; continue
            busy = j * T1 + C1s[j % 65536]
            if t < busy: t = busy
            nxt = (j + 1) * T1
            if t + w <= nxt: t += w; break
            w -= nxt - t; t = nxt; j += 1
        fin = t; R[i] = t - rel
    return R
miss = lambda R, T2: float((np.asarray(R) > T2).mean())
def block_ci(ind, L=100, B=1000, seed=3):                   # moving-block bootstrap: honest CI when misses are autocorrelated
    ind = np.asarray(ind, float); n = len(ind); nb = -(-n // L)
    st = np.random.default_rng(seed).integers(0, n - L + 1, (B, nb)); idx = (st[:, :, None] + np.arange(L)).reshape(B, -1)[:, :n]
    return [float(v) for v in np.percentile(ind[idx].mean(1), [2.5, 97.5])]
def bisect(feasible, lo, hi, it=18):
    for _ in range(it):
        mid = (lo + hi) / 2; lo, hi = (lo, mid) if feasible(mid) else (mid, hi)
    return hi
def rta_det(C2max, C1max, T2):                              # classic deterministic RTA with measured WCET (max observed)
    T1 = T2 / K
    if C1max >= T1: return np.inf
    R = C2max + C1max
    for _ in range(10000):
        Rn = C2max + math.ceil(R / T1 - 1e-9) * C1max
        if Rn == R or Rn > 50 * T2: return Rn
        R = Rn
    return R

In [ ]:
# 5) Design from the ID profile, then check reality under shift (~20-40 s)
prof = T["ID_profile"]; c1 = 0.15 * np.median(prof)      # control task = 15% of a median inference, K x per frame
lo, hi = 1.05 * K * 1.2 * c1, 4 * prof.max()
T2_prob = bisect(lambda T2: analyse(prof, c1, T2) <= TARGET, lo, hi)          # shortest period with predicted miss <= 1%
T2_wcet = bisect(lambda T2: rta_det(prof.max(), 1.2 * c1, T2) <= T2, lo, hi)  # shortest period that is WCET-schedulable
pred_design = analyse(prof, c1, T2_prob)
print(f"design period: probabilistic {T2_prob:.3f} ms (pred miss {pred_design:.4f}) | WCET {T2_wcet:.3f} ms | saving {100 * (1 - T2_prob / T2_wcet):.1f}%")
DEPLOY = ["ID", "Shift_mild", "Shift_strong", "OOD_noise", "Bursty"]; rows = []
for c in DEPLOY:
    t = T[c]; Ro = simulate(t, c1, T2_prob); Rs = simulate(rng.permutation(t), c1, T2_prob)
    mo = miss(Ro, T2_prob)
    rows.append({"condition": c, "pred_miss_design(ID profile)": pred_design, "analysis_true_PMF_iid": analyse(t, c1, T2_prob),
                 "sim_shuffled_iid": miss(Rs, T2_prob), "sim_real_order": mo, "real_order_CI95_block": block_ci(Ro > T2_prob),
                 "underestimation_x": mo / max(pred_design, 1e-12), "WCET_design_sim_miss": miss(simulate(t, c1, T2_wcet), T2_wcet)})
MT = pd.DataFrame(rows); VAL = float((MT["analysis_true_PMF_iid"] - MT["sim_shuffled_iid"]).abs().max())
print(MT.round(4).to_string(index=False)); print(f"analysis validation: max |analysis(true PMF) - shuffled simulation| = {VAL:.4f}")
grid = np.geomspace(max(0.8 * T2_prob, lo), 1.15 * T2_wcet, 20)
SW = {"T2": grid, "pred": [analyse(prof, c1, x) for x in grid]}
for c in DEPLOY: SW[c] = [miss(simulate(T[c], c1, x), x) for x in grid]

In [ ]:
# 6) Figure
fig, ax = plt.subplots(2, 2, figsize=(12, 8.5)); col = {"ID_profile": "k", "ID": "C0", "Shift_mild": "C1", "Shift_strong": "C3", "OOD_noise": "C4", "Bursty": "C2"}
for c in ["ID_profile"] + DEPLOY:
    x = np.sort(T[c]); ax[0, 0].semilogy(x, 1 - np.arange(len(x)) / len(x), color=col[c], ls="--" if c == "ID_profile" else "-", label=c)
ax[0, 0].set(xlabel="inference time (ms)", ylabel="P(time > t)", title="(a) Execution-time exceedance"); ax[0, 0].legend(fontsize=8)
fr = np.array([[(ST[c] == s).mean() for s in (1, 2, 3)] for c in DEPLOY]); bot = np.zeros(len(DEPLOY))
for s in range(3): ax[0, 1].bar(DEPLOY, fr[:, s], bottom=bot, label=f"exit {s + 1}"); bot += fr[:, s]
ax[0, 1].set(ylabel="fraction of frames", title=f"(b) Exit depth (tau={TAU})"); ax[0, 1].legend(fontsize=8); ax[0, 1].tick_params(axis="x", rotation=20)
fl = 1 / (2 * N)
ax[1, 0].semilogy(SW["T2"], np.maximum(SW["pred"], fl), "k--", lw=2, label="design-time prediction (ID profile, i.i.d.)")
for c in DEPLOY: ax[1, 0].semilogy(SW["T2"], np.maximum(SW[c], fl), "o-", ms=3, color=col[c], label=f"simulated: {c}")
ax[1, 0].axvline(T2_prob, color="gray", ls=":", label="probabilistic design"); ax[1, 0].axvline(T2_wcet, color="gray", ls="-.", label="WCET design")
ax[1, 0].axhline(TARGET, color="r", lw=.8); ax[1, 0].set(xlabel="period = deadline T2 (ms)", ylabel="deadline-miss probability (floor = none observed)", title="(c) Predicted vs real miss rate"); ax[1, 0].legend(fontsize=7)
Ro = simulate(T["Bursty"], c1, T2_prob); idx = np.arange(N); top = T["Bursty"].max()
ax[1, 1].plot(idx, T["Bursty"], lw=.5, color="C2", label="execution time"); ax[1, 1].fill_between(idx, 0, top, where=state == 1, color="orange", alpha=.25, label="shift episode")
ax[1, 1].plot(idx[Ro > T2_prob], np.full(int((Ro > T2_prob).sum()), top), "rv", ms=3, label="deadline miss")
ax[1, 1].set(xlabel="frame", ylabel="ms", title="(d) Bursty shift: misses cluster inside episodes"); ax[1, 1].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.savefig(f"{OUT}/pilot_figure.png", dpi=160); plt.show()

In [ ]:
# 7) Save everything + GitHub README + zip + the block to paste back
SUMMARY = {"env": ENV, "config": {"N_per_condition": N, "TAU": TAU, "K": K, "target_miss": TARGET, "c1_ms": c1, "bins_per_T1": NB},
           "accuracy_batch": ACC, "timing_ms": TS.to_dict(orient="index"), "strong_shift_static": STATIC, "strong_shift_early_exit": EE,
           "design": {"T2_prob_ms": T2_prob, "T2_wcet_ms": T2_wcet, "saving_pct": 100 * (1 - T2_prob / T2_wcet), "pred_miss": pred_design},
           "miss_table": MT.to_dict(orient="records"), "analysis_validation_max_abs_err": VAL}
J = lambda o: o.item() if hasattr(o, "item") else (o.tolist() if hasattr(o, "tolist") else str(o))
json.dump(SUMMARY, open(f"{OUT}/summary.json", "w"), indent=2, default=J)
MT.to_csv(f"{OUT}/miss_table.csv", index=False); TS.to_csv(f"{OUT}/timing_stats.csv")
np.savez_compressed(f"{OUT}/timings.npz", bursty_state=state, **{f"time_{c}": T[c] for c in CONDS}, **{f"exit_{c}": ST[c] for c in CONDS})
torch.save(m.state_dict(), f"{OUT}/early_exit_cnn.pt")
r = {row["condition"]: row for row in MT.to_dict(orient="records")}
tab = "\n".join(f"| {c} | {TS.loc[c, 'exit3']:.2f} | {TS.loc[c, 'p99']:.3f} | {r[c]['pred_miss_design(ID profile)']:.3f} | {r[c]['sim_real_order']:.3f} | {r[c]['sim_shuffled_iid']:.3f} |" for c in DEPLOY)
open(f"{REPO}/README.md", "w").write(f"""# Timing reliability of early-exit inference under distribution shift (pilot)

Pilot for the MSc proposal *Characterising and Evaluating the Timing Reliability of Embedded Neural-Network Inference under Probabilistic Real-Time Scheduling*.

**Question.** An early-exit CNN is profiled on in-distribution (ID) data; a convolution-based probabilistic response-time analysis picks the shortest period whose predicted deadline-miss probability is <= {TARGET:.0%}. What is the real miss rate when the inputs shift?

**Setup.** 3-exit CNN on scikit-learn digits (8x8 upsampled to 32x32), exit threshold {TAU}. One CPU thread ({ENV['cpu']}), {N} timed inferences per condition, conditions interleaved in random order. One core, fixed-priority preemptive: control task ({K}x per frame, exec U[0.8,1.2]x{c1:.3f} ms) above the inference task (period = deadline). Analysis: stationary-backlog, split-convolve-merge stochastic response-time analysis (i.i.d. execution times). Ground truth: trace-driven simulation of the measured execution-time sequence.

## Results
Design period from the ID profile: **{T2_prob:.3f} ms** (predicted miss {pred_design:.3f}) vs **{T2_wcet:.3f} ms** for a WCET (max-observed) design ({100 * (1 - T2_prob / T2_wcet):.0f}% shorter).

| condition | frames at full depth | p99 latency (ms) | predicted miss (design) | real miss (real order) | real miss (shuffled) |
|---|---|---|---|---|---|
{tab}

Static (no-exit) control under strong shift: median latency {STATIC['median_change_pct']:+.1f}% (KS D = {STATIC['KS_D']:.3f}); early-exit network: {EE['median_change_pct']:+.1f}% (KS D = {EE['KS_D']:.3f}). Analysis validation: max |analysis - shuffled simulation| = {VAL:.4f}.

![pilot figure](results/pilot_figure.png)

## Limitations
Toy dataset and model, cloud CPU instead of an embedded board, simplified uniprocessor analysis (not a reproduction of the TII 2026 CAN analysis), single run.

## Run
Open `timing_reliability_pilot.ipynb` in Google Colab, then Runtime -> Run all (about 3 minutes).
""")
shutil.make_archive(REPO, "zip", REPO)
try:
    from google.colab import files; files.download(f"{REPO}.zip")
except Exception:
    print(f"saved {REPO}.zip")
print("\n" + "=" * 25 + " COPY EVERYTHING BELOW AND PASTE IT BACK TO CLAUDE " + "=" * 25)
print("ENV:", ENV); print("ACCURACY (batch):"); print(pd.DataFrame(ACC).T.round(3).to_string())
print("TIMING (ms):"); print(TS[["mean", "p50", "p99", "p99.9", "max", "exit1", "exit2", "exit3", "acc", "lag1_autocorr", "KS_D_vs_profile"]].round(3).to_string())
print("STRONG SHIFT -> static:", {k: round(float(v), 3) for k, v in STATIC.items()}, "| early-exit:", {k: round(float(v), 3) for k, v in EE.items()})
print(f"DESIGN: T2_prob={T2_prob:.3f} ms pred_miss={pred_design:.4f} | T2_wcet={T2_wcet:.3f} ms | saving={100 * (1 - T2_prob / T2_wcet):.1f}% | c1={c1:.4f} ms")
print("MISS TABLE:"); print(MT.round(4).to_string(index=False)); print(f"VALIDATION max abs err = {VAL:.4f}"); print("=" * 100)